# 01 - Signal (Q2)

Text -> target with product/vendor tokens masked, against a majority-class baseline.

**Pass (pre-registered `gates.q2`):** masked macro-F1 >= majority + 0.15 (stratified 5-fold).
If the vendor-grouped split keeps < +0.05 over majority, the signal is vendor-specific.

In [ ]:
import pandas as pd
from src.config import load_config, data_path

pd.set_option("display.width", 140)
cfg = load_config()
interim, processed = data_path(cfg, "interim"), data_path(cfg, "processed")

In [ ]:
import json

results = {t: json.loads((processed / f"results_{t}.json").read_text()) for t in cfg["targets"]}
rows = []
for target, r in results.items():
    for name, run in r["signal"].items():
        rows.append({"target": target, "run": name, "macro_f1": run["macro_f1_mean"],
                     "sd": run["macro_f1_sd"], "accuracy": run["accuracy_mean"], "n": run["n"]})
pd.DataFrame(rows).round(3)

## Masking example

In [ ]:
feats = pd.read_parquet(processed / "rows_sector.parquet")
for _, r in feats.sample(3, random_state=1).iterrows():
    print(r["cve_id"], r["label"]); print("  raw:   ", r["description"][:300]); print("  masked:", r["text_masked"][:300]); print()

## Per-class (masked text, stratified)

In [ ]:
for target, r in results.items():
    print(target)
    display(pd.DataFrame(r["signal"]["masked"]["per_class"]).T.round(3))

## Q2 decision

In [ ]:
{t: r['q2'] for t, r in results.items()}